In [1]:
# This script processes potential duplicated stations using produced subset of distance matrix containing only stations which have at least one other station in range < threshold 
# distance (saved in output_data: duplicated_only_distmatrix_0.25km.pkl)

# 1. read in filtered merged water quality raw data and merged station data and calculate some stats to get an overview of available stations
# 2. read in by threshold subsetted distance matrix to filter potential_duplicated "station_pairs"
# 3. calculate a list of all potential duplicated stations (station pairs with distance < threshold) from subsetted distance matrix
# 4. filter this list and exclude pairs where stations are paired with themself e.g. GRQA Id_01 and GRQA Id_01 -->  have a distance of 0.0km 

# 5. read in again merged station data: but now assigned to stream_ids from HydroRiver shapefile (this file is produced  using notebook "p_06_assign_stations_to_HydroRivers.ipynb"
# and can be find in cnp-synthesis\output_data\assign_stations_HydroRivers\ArcPy_output
# 6. Merge pairs of potential duplicated stations and stations dataset containing infos about countries, dataset, assigned river_id
# 7. save list of all pot_duplicated_stations with all the added infos as csv and additionally a list of station pairs with different countries as well 

In [2]:
# load required modules:

import pandas as pd
from glob import glob
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import warnings
from tqdm import trange
from time import time
import pickle
from scipy.spatial.distance import cdist
from scipy.spatial import distance_matrix
import pdb
from math import sin, cos, sqrt, atan2, radians
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics.pairwise import pairwise_distances
import multiprocessing
from multiprocessing import Pool
n_cores = multiprocessing.cpu_count()
import os
from tqdm import tqdm
from time import time

In [14]:
# define the output folder path:

output_folder = '../../output_data/merge_0km_dist_stations'

# Check if the folder exists, if not, create it:
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

 ## 1. read in merged water quality raw data and merged station data and calculate some stats to get an overview of available stations

In [4]:
#--- Stations Data ---
tic = time()
stations = gpd.read_file('../../output_data/merged_datasets/stations.shp')
stations = stations.set_index(['dataset','site_id']).sort_index()
stations = stations.reset_index()
stations['site_id'] = stations['site_id'].astype(str)
stations = stations.set_index(['dataset','site_id'])
print('Block time [seg]: ',f'{time()-tic}')

Block time [seg]:  0.9361817836761475


In [5]:
#--------------- WATER QUALITY PROCESSING------------ ########################
# read in merged raw data:
tic = time()
raw_data = pd.read_csv('../../output_data/merged_datasets/filtered_merged_raw_data.csv')
raw_data = raw_data.set_index(['dataset', 'site_id']).replace(-9999,np.nan).sort_index()
raw_data['obs_date'] = pd.DatetimeIndex(raw_data['obs_date'].values)
print('Loaded raw data in [sec]: ', f'{time() -tic}')



# ----id's to string----

raw_data = raw_data.reset_index()
raw_data['site_id'] = raw_data['site_id'].astype(str)
raw_data = raw_data.set_index(['dataset', 'site_id'])


# ---change the Ob' of arcticdeltas data
raw_data = raw_data.reset_index()
raw_data['site_id'] = raw_data['site_id'].replace('Ob\'','Ob').values

# get sorted index
raw_data = raw_data.set_index(['dataset', 'site_id']).sort_index()
raw_data_index = raw_data.sort_index().index.unique()
print("Block time [sec]: ",f'{time()-tic}')


C:\Users\bartusch\AppData\Local\Temp\ipykernel_4612\3324177926.py:4: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv('../../output_data/merged_datasets/filtered_merged_raw_data.csv')


Loaded raw data in [sec]:  9.244240999221802
Block time [sec]:  12.822776317596436


In [6]:
#--- Obtain stats of compounds we are interested per station ---
tic = time()
stations_index = stations.index
output = []
#compounds = raw_data.columns.to_list()[4:]
compounds = ['NH4N','NO3N','NO2N','TOC','DOC','TP','DIP', 'OPO4']
for comp in compounds:
    print(comp)
    raw_data[comp] = raw_data[comp].astype(float).values
    aux_data = raw_data[[comp,'obs_date']].copy().dropna()
    group = aux_data.groupby(level=[0,1])
    median = group.median()[[comp]].rename(columns={comp:f'{comp}_median'})
    q25 = group.quantile(0.25)[[comp]].rename(columns={comp:f'{comp}_25'})
    q75 = group.quantile(0.75)[[comp]].rename(columns={comp:f'{comp}_75'})
    n = group.count()[[comp]].rename(columns={comp:f'{comp}_n'})
    start = group.min()[['obs_date']].rename(columns={'obs_date':f'{comp}_start'})
    end = group.max()[['obs_date']].rename(columns={'obs_date':f'{comp}_end'})
    stats = pd.concat([median,q25,q75,n,start,end],axis=1)
    cols = stats.columns
    output.append(stats.copy())
stations_stats = pd.concat(output,axis=1)
print("Elapsed %.2f seconds" % ((time() - tic)))

NH4N
NO3N
NO2N
TOC
DOC
TP
DIP
OPO4
Elapsed 4.25 seconds


In [7]:
#print(stations_stats)

## Set threshold: distance pot duplicated



In [8]:
thrs = 0.25 # else 0.5

## 2. Read in subsetted distance matrix (adapt filename corresponding to threshold)

In [10]:
# subset of distance matrix:
# ---load distance matrix ----
tic = time()
stations_dup_km_2 = pickle.load(open(f'../../output_data/duplicated_only_distmatrix_{thrs}km.pkl', 'rb'))
print("Elapsed %.2f seconds" % ((time() - tic)))
#print(dist_matrix.shape,stations_stats.shape)        

Elapsed 1.41 seconds


## 3. calculate a list of all potential duplicated stations (station pairs with distance < threshold) from subsetted distance matrix

In [11]:
# loop over all columns of subsetted distance matrix to get all duplicates:
cols = stations_dup_km_2.columns

all_A = []
all_B = []
all_distances = []
for i in cols:
    inds_thrs = stations_dup_km_2[stations_dup_km_2[i]<=thrs].index.to_list()
    # select rows of stations with distance smmaller threshold to station in column i
    rows_distance = stations_dup_km_2[(stations_dup_km_2[i] <= thrs)]
    # Extract the values from column 'i' from filtered rows containing distances<threshold
    distances = rows_distance[i]
    length = len(inds_thrs)
    station_A = [i] * length
    station_B = inds_thrs
    all_A.extend(station_A)
    all_B.extend(station_B)
    all_distances.extend(distances)
# organise in a dataframe
pot_duplicated_stations = pd.DataFrame({'station_id_a':all_A, 'station_id_duplic':all_B, 'distance_km':all_distances})
# define data types
pot_duplicated_stations['station_id_a'] = pot_duplicated_stations['station_id_a'].astype(str)
pot_duplicated_stations['station_id_duplic'] = pot_duplicated_stations['station_id_duplic'].astype(str)
pot_duplicated_stations['distance_km'] = pot_duplicated_stations['distance_km'].astype(str)


In [12]:
print(pot_duplicated_stations) 
# 69966 rows in pot_duplicated_stations (threshold 0.5km)--> but has to be filtered
# 45356 rows in pot_duplicated_stations (threshold 0.25km)--> but has to be filtered

             station_id_a          station_id_duplic distance_km
0      ('GRQA', '100002')         ('GRQA', '100002')         0.0
1      ('GRQA', '100002')  ('GRQA', 'USGS-01017100')       0.147
2      ('GRQA', '100003')         ('GRQA', '100003')         0.0
3      ('GRQA', '100003')  ('GRQA', 'USGS-01017500')       0.036
4      ('GRQA', '100004')         ('GRQA', '100004')         0.0
...                   ...                        ...         ...
45351    ('sweden', '66')           ('sweden', '60')      0.1613
45352    ('sweden', '66')           ('sweden', '66')         0.0
45353     ('sweden', '7')            ('sweden', '2')      0.0146
45354     ('sweden', '7')            ('sweden', '6')      0.2327
45355     ('sweden', '7')            ('sweden', '7')         0.0

[45356 rows x 3 columns]


## 4. Filter list of  "station pairs" --> stations that are potential duplicates

In [13]:
# -----create subset of all stations that are potential duplicates: ----- 

# this line excludes rows from dataframe if both are same station id's: e.g.: 'GRQA', '100586' and 'GRQA', '100586'
duplicated_stations_3 = pot_duplicated_stations.iloc[pot_duplicated_stations[pot_duplicated_stations['station_id_duplic']!=pot_duplicated_stations['station_id_a']].index,:]
print(duplicated_stations_3)
print(duplicated_stations_3.info())

# next step: each pair of potential duplicated stations_id with distance of 0km occurs two times in dataframe
# these duplicates must be removed:

# convert station id's to strings
combined_correct = duplicated_stations_3['station_id_a'].astype(str) + duplicated_stations_3['station_id_duplic'].to_list()
combined_inverted = duplicated_stations_3['station_id_duplic'].astype(str) + duplicated_stations_3['station_id_a'].to_list()

# filter only one version of all station combinations

duplicated_stations_4 = duplicated_stations_3.copy()
duplicated_stations_4.columns = ['x','y', 'distance_km']
print(duplicated_stations_4.columns)

duplicated_stations_4.loc[:,'filter'] = duplicated_stations_4.loc[:,['x','y']].apply(lambda row: '-'.join(sorted(row, key=lambda char: char)), axis=1)
 # save all pairs to                                                                     
pairs_pot_duplicated = duplicated_stations_4.drop_duplicates(subset=['filter']).reset_index()[['x','y','distance_km']]
# contains 48516 pot duplicated station pairs (threshold 0.5km)

             station_id_a          station_id_duplic distance_km
1      ('GRQA', '100002')  ('GRQA', 'USGS-01017100')       0.147
3      ('GRQA', '100003')  ('GRQA', 'USGS-01017500')       0.036
5      ('GRQA', '100004')         ('GRQA', '110876')      0.0124
6      ('GRQA', '100004')         ('GRQA', '110877')      0.0124
7      ('GRQA', '100004')       ('GRQA', 'CAN00052')      0.1286
...                   ...                        ...         ...
45348    ('sweden', '61')           ('sweden', '62')      0.1192
45349    ('sweden', '62')           ('sweden', '61')      0.1192
45351    ('sweden', '66')           ('sweden', '60')      0.1613
45353     ('sweden', '7')            ('sweden', '2')      0.0146
45354     ('sweden', '7')            ('sweden', '6')      0.2327

[30130 rows x 3 columns]
<class 'pandas.core.frame.DataFrame'>
Index: 30130 entries, 1 to 45354
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----

#### Save data:

In [15]:

# print info of pot_duplicated stations: 
print(pairs_pot_duplicated.info())
print(pairs_pot_duplicated)
# save file to csv: 
pairs_pot_duplicated.to_csv(f'../../output_data/merge_0km_dist_stations/all_pairs_of_potential_dup_stations_{thrs}km.csv')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15065 entries, 0 to 15064
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   x            15065 non-null  object
 1   y            15065 non-null  object
 2   distance_km  15065 non-null  object
dtypes: object(3)
memory usage: 353.2+ KB
None
                        x                          y distance_km
0      ('GRQA', '100002')  ('GRQA', 'USGS-01017100')       0.147
1      ('GRQA', '100003')  ('GRQA', 'USGS-01017500')       0.036
2      ('GRQA', '100004')         ('GRQA', '110876')      0.0124
3      ('GRQA', '100004')         ('GRQA', '110877')      0.0124
4      ('GRQA', '100004')       ('GRQA', 'CAN00052')      0.1286
...                   ...                        ...         ...
15060     ('sweden', '2')            ('sweden', '7')      0.0146
15061    ('sweden', '58')           ('sweden', '59')       0.227
15062     ('sweden', '6')            ('sweden', '7')   

In [16]:
# Get number of different Station_id's in x and y: 

unique_station_ids = pd.concat([pairs_pot_duplicated['x'], pairs_pot_duplicated['y']]).unique()
print(f'A number of, {len(unique_station_ids)} stations has duplicated stations in distance of {thrs} km')


# get number n of stations that are at same location : How many stations are at same location? für jede Station x wie oft kommt sie vor,
# also wie viele Stationen sind an der Position 
all_occurences = pd.concat([pairs_pot_duplicated['x'], pairs_pot_duplicated['y']])
#all_occurences.groupby('category')['value'].transform('sum')

all_o = pd.DataFrame({'ocurrences':all_occurences})
menge = all_o.groupby("ocurrences").size().reset_index(name="Anzahl")
print(menge)
# maximum 49 stations at same position GRQA NL05--> example, would be interesting to know what is different ?
menge[menge['Anzahl']==49]
#menge.groupby('Anzahl').count().sort('occurences')

# A number of, 21450 stations has duplicated stations in distance of 0.5km
# A number of, 15226 stations has duplicated stations in distance of 0.25 km

A number of, 15226 stations has duplicated stations in distance of 0.25 km
               ocurrences  Anzahl
0      ('GRQA', '100002')       1
1      ('GRQA', '100003')       1
2      ('GRQA', '100004')       6
3      ('GRQA', '100005')       1
4      ('GRQA', '100007')       1
...                   ...     ...
15221    ('sweden', '60')       1
15222    ('sweden', '61')       1
15223    ('sweden', '62')       1
15224    ('sweden', '66')       1
15225     ('sweden', '7')       2

[15226 rows x 2 columns]


,ocurrences,Anzahl
3706,"('GRQA', 'NL05_01_009')",49
3707,"('GRQA', 'NL05_01_026')",49
3708,"('GRQA', 'NL05_01_037')",49
3711,"('GRQA', 'NL05_02_006')",49
3713,"('GRQA', 'NL05_02_037')",49
3714,"('GRQA', 'NL05_02_039')",49
3715,"('GRQA', 'NL05_02_046')",49
3716,"('GRQA', 'NL05_02_049')",49
3718,"('GRQA', 'NL05_02_203')",49
3719,"('GRQA', 'NL05_02_302')",49


## 5. Read in again merged station data, but now assigned to stream id'S from HydroRivers shapefile: 

In [18]:
# read in again station data with added information about closest river id in HydroRivers shapefile
# infromation can be found in columns NEAR_FID and the distance in NEAR_DIST 

stations = gpd.read_file('../../output_data/assign_stations_HydroRivers/ArcPy_output/stations_assigned_river.shp')
stations['SITE_id'] = stations['site_id']
stations['data_set'] = stations['dataset']
stations = stations.set_index(['dataset','site_id']).sort_index()


print(stations)

                               data_sourc country          area        lat  \
dataset site_id                                                              
GRQA    100001                    GLORICH     USA  2.120184e+10  47.160181   
        100002                    GLORICH     USA  4.994408e+09  46.847909   
        100003                    GLORICH     USA  5.949347e+09  46.773619   
        100004                    GLORICH     USA  3.782731e+09  45.168718   
        100005                    GLORICH     USA  5.736990e+08  44.606084   
...                                   ...     ...           ...        ...   
sweden  64       https://data.krycklan.se  sweden -9.999000e+03  64.162568   
        65       https://data.krycklan.se  sweden -9.999000e+03  64.175916   
        66       https://data.krycklan.se  sweden -9.999000e+03  64.172863   
        7        https://data.krycklan.se  sweden -9.999000e+03  64.251301   
        9        https://data.krycklan.se  sweden -9.999000e+03 

## 6. Merge pairs of potential duplicated stations and stations dataset containing infos about countries, dataset, assigned river_id 

In [19]:
###add criteria, in order to check whether station coordinates, country, river id (HydroRivers) and dataset are same or different 

######################################################################################################

# next step: Add information to extracted data table

# modify  station data:
station_2 = stations.copy()
station_2['station_id'] = station_2.index
station_2['station_id'] =station_2['station_id'].astype(str)
station_2 = station_2.reset_index()


merged_data = pairs_pot_duplicated.merge(station_2[['station_id', 'data_sourc', 'country', 'area', 'lat', 'lon', 'NEAR_FID', 'NEAR_DIST']], left_on='x', right_on='station_id', how='left')

# add prefix x_ to column names:
new_column_names_x = merged_data.columns[:3].tolist() + ['x_' + col for col in merged_data.columns[3:]]
merged_data.columns = new_column_names_x
print(merged_data)

# add data of y stations to dataframe:
all_pairs_dup = merged_data.merge(station_2[['station_id', 'data_sourc', 'country', 'area', 'lat', 'lon', 'NEAR_FID', 'NEAR_DIST']], left_on='y', right_on='station_id', how='left')

# add prefix y_ to column names:
cols_renamey = 8
new_column_names_y = all_pairs_dup.columns[:-cols_renamey].tolist() + ['y_' + col for col in all_pairs_dup.columns[-cols_renamey:]]
all_pairs_dup.columns = new_column_names_y
all_pairs_dup



                        x                          y distance_km  \
0      ('GRQA', '100002')  ('GRQA', 'USGS-01017100')       0.147   
1      ('GRQA', '100003')  ('GRQA', 'USGS-01017500')       0.036   
2      ('GRQA', '100004')         ('GRQA', '110876')      0.0124   
3      ('GRQA', '100004')         ('GRQA', '110877')      0.0124   
4      ('GRQA', '100004')       ('GRQA', 'CAN00052')      0.1286   
...                   ...                        ...         ...   
15060     ('sweden', '2')            ('sweden', '7')      0.0146   
15061    ('sweden', '58')           ('sweden', '59')       0.227   
15062     ('sweden', '6')            ('sweden', '7')      0.2327   
15063    ('sweden', '60')           ('sweden', '66')      0.1613   
15064    ('sweden', '61')           ('sweden', '62')      0.1192   

             x_station_id              x_data_sourc x_country        x_area  \
0      ('GRQA', '100002')                   GLORICH       USA  4.994408e+09   
1      ('GRQA', '100003')

,x,y,distance_km,x_station_id,x_data_sourc,x_country,x_area,x_lat,x_lon,x_NEAR_FID,x_NEAR_DIST,y_station_id,y_data_sourc,y_country,y_area,y_lat,y_lon,y_NEAR_FID,y_NEAR_DIST
0,"('GRQA', '100002')","('GRQA', 'USGS-01017100')",0.147,"('GRQA', '100002')",GLORICH,USA,4.994408e+09,46.847909,-68.002459,7396181,28.635691,"('GRQA', 'USGS-01017100')",WQP,United States,1.943000e+03,46.849209,-68.002804,7396181,55.003173
1,"('GRQA', '100003')","('GRQA', 'USGS-01017500')",0.036,"('GRQA', '100003')",GLORICH,USA,5.949347e+09,46.773619,-67.831939,7398300,0.795138,"('GRQA', 'USGS-01017500')",WQP,United States,2.301000e+03,46.773372,-67.831633,7398300,4.542234
2,"('GRQA', '100004')","('GRQA', '110876')",0.0124,"('GRQA', '100004')",GLORICH,USA,3.782731e+09,45.168718,-67.298037,7453269,10.131314,"('GRQA', '110876')",GLORICH,Canada,3.782731e+09,45.168609,-67.298070,7453269,18.885529
3,"('GRQA', '100004')","('GRQA', '110877')",0.0124,"('GRQA', '100004')",GLORICH,USA,3.782731e+09,45.168718,-67.298037,7453269,10.131314,"('GRQA', '110877')",GLORICH,Canada,3.782731e+09,45.168609,-67.298070,7453269,18.885529
4,"('GRQA', '100004')","('GRQA', 'CAN00052')",0.1286,"('GRQA', '100004')",GLORICH,USA,3.782731e+09,45.168718,-67.298037,7453269,10.131314,"('GRQA', 'CAN00052')",GEMSTAT,Canada,-9.999000e+03,45.169720,-67.297220,7453269,17.538434
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15060,"('sweden', '2')","('sweden', '7')",0.0146,"('sweden', '2')",https://data.krycklan.se,sweden,-9.999000e+03,64.251425,19.777804,1616828,2249.110325,"('sweden', '7')",https://data.krycklan.se,sweden,-9.999000e+03,64.251301,19.777900,1616828,2235.286633
15061,"('sweden', '58')","('sweden', '59')",0.227,"('sweden', '58')",https://data.krycklan.se,sweden,-9.999000e+03,64.179048,19.863411,1618649,462.150228,"('sweden', '59')",https://data.krycklan.se,sweden,-9.999000e+03,64.177068,19.864548,1618649,406.903617
15062,"('sweden', '6')","('sweden', '7')",0.2327,"('sweden', '6')",https://data.krycklan.se,sweden,-9.999000e+03,64.250974,19.773143,1616828,2198.841702,"('sweden', '7')",https://data.krycklan.se,sweden,-9.999000e+03,64.251301,19.777900,1616828,2235.286633
15063,"('sweden', '60')","('sweden', '66')",0.1613,"('sweden', '60')",https://data.krycklan.se,sweden,-9.999000e+03,64.172828,19.866116,1618649,492.794143,"('sweden', '66')",https://data.krycklan.se,sweden,-9.999000e+03,64.172863,19.862788,1618649,639.563847


In [20]:
# add country code: if both from same country yes else no
# in dataset occure some aliases: for example: PL and poland or NL and Netherlands

# Define a dictionary of country aliases
country_aliases = {
    'Poland': ['PL'],
    'USA': ['US', 'United States', 'United States of America (the)'],
    'Sweden': ['SE', 'sweden', 'SWEDEN'],
    'Netherlands':['NL', 'Netherlands (the)'],
    'Austria':['AT'],
    'Switzerland':['CH'],
    'South Africa':['RSA'],
    'Croatia':['HR'],
    'UK': ['United Kingdom of Great Britain and Northern Ireland (the)'],
    'Macedonia (the former Yugoslav Republic of)':['MK']
}

# Function to replace values based on aliases
def replace_with_alias(value):
    for country, aliases in country_aliases.items():
        if value in aliases:
            return country
    return value

# Apply the replacement function to columns 'x' and 'y'
all_pairs_dup['x_country'] = all_pairs_dup['x_country'].apply(replace_with_alias)
all_pairs_dup['y_country'] = all_pairs_dup['y_country'].apply(replace_with_alias)

# add column 'same country' to dataframe: if both stations are from same country add True else False
all_pairs_dup['same_country'] = np.where(all_pairs_dup['x_country']==all_pairs_dup['y_country'],True, False)
print(all_pairs_dup[all_pairs_dup['same_country']==False])
diff_country  =all_pairs_dup[all_pairs_dup['same_country']==False]

# save file containing list of pot_duplicated stations origin from different countries
diff_country.to_csv(f'../../output_data/merge_0km_dist_stations/diff_country_code_{thrs}_km.csv')

diff_country

# add dataset
# add observed compounds for x and y --> überlappung pder ergänzend

                               x                             y distance_km  \
2             ('GRQA', '100004')            ('GRQA', '110876')      0.0124   
3             ('GRQA', '100004')            ('GRQA', '110877')      0.0124   
4             ('GRQA', '100004')          ('GRQA', 'CAN00052')      0.1286   
5             ('GRQA', '100004')          ('GRQA', 'CAN00200')      0.2014   
1499          ('GRQA', '102516')          ('GRQA', 'MEX03538')      0.1939   
1500          ('GRQA', '102516')          ('GRQA', 'MEX03539')      0.1906   
1613          ('GRQA', '102703')            ('GRQA', '110088')      0.0112   
2286          ('GRQA', '110047')     ('GRQA', 'USGS-12211900')      0.0924   
2292          ('GRQA', '110088')          ('USGS', '12355000')      0.1855   
2305          ('GRQA', '110685')     ('GRQA', 'USGS-01014000')      0.2216   
2310          ('GRQA', '110694')     ('GRQA', 'USGS-01014000')      0.2216   
2312          ('GRQA', '110695')     ('GRQA', 'USGS-01014000')  

,x,y,distance_km,x_station_id,x_data_sourc,x_country,x_area,x_lat,x_lon,x_NEAR_FID,x_NEAR_DIST,y_station_id,y_data_sourc,y_country,y_area,y_lat,y_lon,y_NEAR_FID,y_NEAR_DIST,same_country
2,"('GRQA', '100004')","('GRQA', '110876')",0.0124,"('GRQA', '100004')",GLORICH,USA,3.782731e+09,45.168718,-67.298037,7453269,10.131314,"('GRQA', '110876')",GLORICH,Canada,3.782731e+09,45.168609,-67.298070,7453269,18.885529,False
3,"('GRQA', '100004')","('GRQA', '110877')",0.0124,"('GRQA', '100004')",GLORICH,USA,3.782731e+09,45.168718,-67.298037,7453269,10.131314,"('GRQA', '110877')",GLORICH,Canada,3.782731e+09,45.168609,-67.298070,7453269,18.885529,False
4,"('GRQA', '100004')","('GRQA', 'CAN00052')",0.1286,"('GRQA', '100004')",GLORICH,USA,3.782731e+09,45.168718,-67.298037,7453269,10.131314,"('GRQA', 'CAN00052')",GEMSTAT,Canada,-9.999000e+03,45.169720,-67.297220,7453269,17.538434,False
5,"('GRQA', '100004')","('GRQA', 'CAN00200')",0.2014,"('GRQA', '100004')",GLORICH,USA,3.782731e+09,45.168718,-67.298037,7453269,10.131314,"('GRQA', 'CAN00200')",GEMSTAT,Canada,-9.999000e+03,45.168600,-67.300600,7453269,211.608309,False
1499,"('GRQA', '102516')","('GRQA', 'MEX03538')",0.1939,"('GRQA', '102516')",GLORICH,USA,7.431277e+08,32.664588,-115.502095,7780241,0.478397,"('GRQA', 'MEX03538')",GEMSTAT,Mexico,-9.999000e+03,32.664502,-115.500027,7780241,9.021653,False
1500,"('GRQA', '102516')","('GRQA', 'MEX03539')",0.1906,"('GRQA', '102516')",GLORICH,USA,7.431277e+08,32.664588,-115.502095,7780241,0.478397,"('GRQA', 'MEX03539')",GEMSTAT,Mexico,-9.999000e+03,32.666080,-115.503097,7780241,34.582157,False
1613,"('GRQA', '102703')","('GRQA', '110088')",0.0112,"('GRQA', '102703')",GLORICH,USA,1.124404e+09,49.002090,-114.477097,7319028,0.433733,"('GRQA', '110088')",GLORICH,Canada,1.124404e+09,49.002182,-114.477036,7319028,8.958717,False
2286,"('GRQA', '110047')","('GRQA', 'USGS-12211900')",0.0924,"('GRQA', '110047')",GLORICH,Canada,3.681149e+07,49.002305,-122.406209,7319893,10.898800,"('GRQA', 'USGS-12211900')",WQP,USA,-9.999000e+03,49.002618,-122.407382,7319893,101.718721,False
2292,"('GRQA', '110088')","('USGS', '12355000')",0.1855,"('GRQA', '110088')",GLORICH,Canada,1.124404e+09,49.002182,-114.477036,7319028,8.958717,"('USGS', '12355000')",USGS,USA,1.117890e+03,49.000529,-114.477371,7319483,112.599523,False
2305,"('GRQA', '110685')","('GRQA', 'USGS-01014000')",0.2216,"('GRQA', '110685')",GLORICH,Canada,1.477333e+10,47.281342,-68.585362,7380508,2.344931,"('GRQA', 'USGS-01014000')",WQP,USA,5.929000e+03,47.283333,-68.585278,7380507,10.507193,False


In [21]:
# add column 'same_river_id' to dataframe: if both stations are assigned to same HydroRiver_ID  add True else False
all_pairs_dup['same_river_id'] = np.where(all_pairs_dup['x_NEAR_FID']==all_pairs_dup['y_NEAR_FID'],True, False)
print(all_pairs_dup[all_pairs_dup['same_river_id']==False])
diff_river_id  =all_pairs_dup[all_pairs_dup['same_river_id']==False]

# save file containing list of pot_duplicated stations origin from different countries
diff_river_id.to_csv(f'../../output_data/merge_0km_dist_stations/diff_river_id_{thrs}_km.csv')

                               x                          y distance_km  \
103           ('GRQA', '100168')  ('GRQA', 'USGS-01334500')      0.1097   
107           ('GRQA', '100173')  ('GRQA', 'USGS-01342800')      0.2409   
224           ('GRQA', '100364')  ('GRQA', 'USGS-01470640')      0.2317   
246           ('GRQA', '100413')  ('GRQA', 'USGS-01485000')      0.1543   
270           ('GRQA', '100446')      ('USGS', 'paWQN0229')      0.2445   
...                          ...                        ...         ...   
14960  ('germany', 'ST_2110050')  ('germany', 'ST_2110060')       0.152   
14966  ('germany', 'ST_2110906')  ('germany', 'ST_2110907')      0.1949   
14991   ('germany', 'ST_310030')   ('germany', 'ST_310610')      0.2313   
15017   ('germany', 'ST_410171')   ('germany', 'ST_410175')      0.2194   
15035   ('germany', 'ST_413820')   ('germany', 'ST_413829')      0.2447   

                    x_station_id x_data_sourc x_country        x_area  \
103           ('GRQA', '10

## 7. save lists of potential duplicated stations:

In [22]:
# save files to csv:
all_pairs_dup.to_csv(f'../../output_data/merge_0km_dist_stations/all_pairs_pot_dup{thrs}km_with_infos.csv')
